# Mini-Project Temperature Lab

## Testing your specimens at different temperatures

This notebook is how you do **Part II** of your team's Shoggoth Field Journal. You take three of
your specimens and run them again and again, at different **temperature** settings, and record
what changes.

### What you'll do

1. Paste in a specimen prompt.
2. Run it at five temperatures: **0.01, 0.5, 0.9, 1.2, 1.5**.
3. Run each temperature **three times**.
4. Copy the results into your team's Journal.

That is 15 runs per specimen, and 45 runs in total. Split them across the team so everyone sees
what happens.

**NOTE:** This notebook uses the same small AI you met in Worksheet 2.2. It runs inside the
notebook, so there is no account and no key to set up.

**Your specimen may not reproduce here, and that is fine.** You found it on a much bigger AI,
like ChatGPT, Claude or Gemini. This one is far smaller, so it may answer differently. Part II is not about making the strange behavior come back. It is about watching how
one model's answers to your prompt spread out as the temperature goes up.

## A. Setup

Run this cell first. It takes about a minute the first time, and you only need to run it once
each time you open the notebook.

In [ ]:
#@title Run this cell first (takes about a minute)

import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
import warnings
warnings.filterwarnings('ignore')

MODEL_NAME = "HuggingFaceTB/SmolLM2-360M-Instruct"

# The notebook asks Colab for a T4 GPU. If Colab has none to give, it still works on the CPU,
# just more slowly.
_device = "cuda" if torch.cuda.is_available() else "cpu"
if _device == "cpu":
    print("⚠️  No GPU today, so the AI will answer more slowly.")
    print("    Try Runtime > Change runtime type > T4 GPU, then run this cell again.")
    print("    (Pick T4 GPU, not TPU: this notebook cannot use a TPU.)\n")

print("Loading the AI (about 700 MB, one time only)... ", end="")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).float().to(_device)   # full precision: the charts need it
model.eval()
VOCAB_SIZE = len(tokenizer)
print("Done!")

def _show(token_id):
    """Show a token so that spaces are visible."""
    return repr(tokenizer.decode(token_id))

def _next_logits(text):
    ids = tokenizer(text, return_tensors="pt").input_ids.to(_device)
    with torch.no_grad():
        return model(ids).logits[0, -1, :]

def _is_word_start(token_id):
    """True if this piece begins a new word (a space, a newline, or punctuation)."""
    s = tokenizer.decode(token_id)
    return s == "" or not s[0].isalnum()

def _whole_word(text, token_id, max_extra=4):
    """The AI sometimes predicts a piece of a word, like ' H'. Finish the word with the
    AI's most likely next pieces, so the tables show whole words, like ' Hanoi'."""
    word = tokenizer.decode(token_id)
    if not word.strip() or not word.strip()[0].isalnum():
        return word                      # punctuation stays as it is
    ids = tokenizer(text, return_tensors="pt").input_ids.to(_device)
    seq = torch.cat([ids, torch.tensor([[token_id]], device=_device)], dim=1)
    for _ in range(max_extra):
        with torch.no_grad():
            nxt = model(seq).logits[0, -1].argmax().item()
        if _is_word_start(nxt):
            break
        word += tokenizer.decode(nxt)
        seq = torch.cat([seq, torch.tensor([[nxt]], device=_device)], dim=1)
    return word

# ============================================================
# FUNCTION 0: Ask the AI a question
# ============================================================
def ask_ai(prompt, temperature=1.0, max_words=50):
    """Ask the AI a question and print its answer."""
    messages = [{"role": "user", "content": prompt}]
    enc = tokenizer.apply_chat_template(messages, add_generation_prompt=True,
                                        return_tensors="pt", return_dict=True).to(_device)
    settings = dict(max_new_tokens=max_words, pad_token_id=tokenizer.eos_token_id)
    if temperature <= 0.01:
        settings["do_sample"] = False
    else:
        settings.update(do_sample=True, temperature=temperature, top_k=0, top_p=1.0)
    with torch.no_grad():
        out = model.generate(**enc, **settings)
    answer = tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)
    print(answer.strip())

# ============================================================
# FUNCTION 1: Show next word probabilities
# ============================================================
def show_next_word_probabilities(text, top_k=10):
    """Show what the AI thinks are the most likely next words."""
    probs = F.softmax(_next_logits(text), dim=-1)
    top_probs, top_indices = torch.topk(probs, top_k)

    print(f'Prompt: "{text}"')
    print(f"\nTop {top_k} predictions for the next word:\n")
    for i in range(top_k):
        prob = top_probs[i].item() * 100
        bar = "*" * int(prob / 2) + "." * (50 - int(prob / 2))
        word = repr(_whole_word(text, top_indices[i].item()))
        print(f"  {word:15} {bar} {prob:5.1f}%")

# ============================================================
# FUNCTION 2: Show temperature effect with "other" category
# ============================================================
def show_temperature_effect(text, temperatures=[0.5, 1.0, 2.0], top_k=5):
    """Show how temperature changes the probability distribution."""
    logits = _next_logits(text)
    # Temperature never changes the ORDER of the words, only their probabilities,
    # so the top words can be finished once and reused for every temperature.
    top_ids = torch.topk(logits, top_k).indices.tolist()
    words = {tid: repr(_whole_word(text, tid)) for tid in top_ids}
    print(f'Prompt: "{text}"\n')
    for temp in temperatures:
        probs = F.softmax(logits / temp, dim=-1)
        top_probs, top_indices = torch.topk(probs, top_k)
        other_prob = 100 - top_probs.sum().item() * 100

        print("=" * 55)
        print(f"Temperature {temp}")
        print("=" * 55)
        for i in range(top_k):
            prob = top_probs[i].item() * 100
            bar = "*" * int(prob / 2) + "." * (50 - int(prob / 2))
            print(f"  {words[top_indices[i].item()]:12} {bar} {prob:5.1f}%")
        other_bar = "*" * int(other_prob / 2) + "." * (50 - int(other_prob / 2))
        print(f"  {'[other]':12} {other_bar} {other_prob:5.1f}%  <- {VOCAB_SIZE - top_k:,} other words")
        print()

# ============================================================
# FUNCTION 3: Generate step by step
# ============================================================
def generate_step_by_step(prompt, num_words=8, temperature=1.0):
    """Generate text and show what was chosen at each step."""
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(_device)

    print(f'Starting prompt: "{prompt}"')
    print(f"Temperature: {temperature}\n")
    print("Step-by-step generation:")
    print("=" * 60)

    for step in range(num_words):
        with torch.no_grad():
            logits = model(input_ids).logits[0, -1, :]
        probs = F.softmax(logits / temperature, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        chosen = next_token.item()
        top_probs, top_indices = torch.topk(probs, 3)

        print(f"\nStep {step + 1}: Top 3 options were:")
        for i in range(3):
            tid = top_indices[i].item()
            mark = " <- CHOSEN" if tid == chosen else ""
            print(f"  {_show(tid)}: {top_probs[i].item() * 100:.1f}%{mark}")
        if chosen not in top_indices.tolist():
            print("     ...")
            print("  (many other words...)")
            print("     ...")
            print(f"  {_show(chosen)}: {probs[chosen].item() * 100:.2f}% <- CHOSEN (surprise pick!)")

        input_ids = torch.cat([input_ids, next_token.unsqueeze(0)], dim=1)

    print("\n" + "=" * 60)
    print(f'Final result: "{tokenizer.decode(input_ids[0], skip_special_tokens=True)}"')

print("\n" + "=" * 50)
print("The AI is ready! Functions available:")
print("  - ask_ai(prompt, temperature=1.0)")
print("  - show_next_word_probabilities(text)")
print("  - show_temperature_effect(text)")
print("  - generate_step_by_step(prompt, temperature=1.0)")
print("  - new_chat() / say(message) / show_chat()")
print("=" * 50)

# ============================================================
# FUNCTION 4: Multi-turn chat (the AI remembers the conversation)
# ============================================================
_chat_history = []

def new_chat():
    """Start a fresh conversation. The AI forgets everything said before."""
    global _chat_history
    _chat_history = []
    print("New chat started.")

def say(message, temperature=1.0, max_words=60):
    """Say something to the AI. It remembers everything earlier in this chat."""
    global _chat_history
    _chat_history = _chat_history + [{"role": "user", "content": message}]
    enc = tokenizer.apply_chat_template(_chat_history, add_generation_prompt=True,
                                        return_tensors="pt", return_dict=True).to(_device)
    settings = dict(max_new_tokens=max_words, pad_token_id=tokenizer.eos_token_id)
    if temperature <= 0.01:
        settings["do_sample"] = False
    else:
        settings.update(do_sample=True, temperature=temperature, top_k=0, top_p=1.0)
    with torch.no_grad():
        out = model.generate(**enc, **settings)
    reply = tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    _chat_history = _chat_history + [{"role": "assistant", "content": reply}]
    print("You:", message)
    print("AI:", reply)

def show_chat():
    """Print the whole conversation so far, to copy into your Journal."""
    print("=" * 50)
    print("FULL CONVERSATION")
    print("=" * 50)
    for turn in _chat_history:
        who = "You" if turn["role"] == "user" else "AI"
        print(f"\n{who}: {turn['content']}")

## B. Quick review from Worksheet 2.2

To send the AI a message, you use `ask_ai`:

In [ ]:
ask_ai("What is Quantitative Reasoning? Respond in two sentences.")

To change the temperature, you add it after the prompt:

In [ ]:
ask_ai("What is Quantitative Reasoning? Respond in two sentences.", temperature=1.2)

Low temperature gives you nearly the same answer every time. High temperature gives you more
variety, more mistakes, and eventually nonsense. That is what you are recording.

## C. Test your specimens

For each specimen: paste your prompt into Step 1, then run each temperature cell **three times**,
and copy the results into your Journal as you go.

### Specimen 1

**Step 1:** Paste your prompt.

In [ ]:
specimen_1_prompt = """
[PASTE YOUR SPECIMEN 1 PROMPT HERE]

Respond in 2-3 sentences.
"""

**Step 2:** Run each of these cells three times.

In [ ]:
ask_ai(specimen_1_prompt, temperature=0.01)

In [ ]:
ask_ai(specimen_1_prompt, temperature=0.5)

In [ ]:
ask_ai(specimen_1_prompt, temperature=0.9)

In [ ]:
ask_ai(specimen_1_prompt, temperature=1.2)

In [ ]:
ask_ai(specimen_1_prompt, temperature=1.5)

### Specimen 2

**Step 1:** Paste your prompt.

In [ ]:
specimen_2_prompt = """
[PASTE YOUR SPECIMEN 2 PROMPT HERE]

Respond in 2-3 sentences.
"""

**Step 2:** Run each of these cells three times.

In [ ]:
ask_ai(specimen_2_prompt, temperature=0.01)

In [ ]:
ask_ai(specimen_2_prompt, temperature=0.5)

In [ ]:
ask_ai(specimen_2_prompt, temperature=0.9)

In [ ]:
ask_ai(specimen_2_prompt, temperature=1.2)

In [ ]:
ask_ai(specimen_2_prompt, temperature=1.5)

### Specimen 3

**Step 1:** Paste your prompt.

In [ ]:
specimen_3_prompt = """
[PASTE YOUR SPECIMEN 3 PROMPT HERE]

Respond in 2-3 sentences.
"""

**Step 2:** Run each of these cells three times.

In [ ]:
ask_ai(specimen_3_prompt, temperature=0.01)

In [ ]:
ask_ai(specimen_3_prompt, temperature=0.5)

In [ ]:
ask_ai(specimen_3_prompt, temperature=0.9)

In [ ]:
ask_ai(specimen_3_prompt, temperature=1.2)

In [ ]:
ask_ai(specimen_3_prompt, temperature=1.5)

## D. Conversations, if your specimen needs one

Some specimens come from a back and forth, not from one message. Use this part only if yours does.

`new_chat()` starts a fresh conversation, `say()` sends one message, and the AI remembers
everything said earlier in that chat.

In [ ]:
new_chat()
say("Hello! I'd like to play a word game with you.")

In [ ]:
say("[YOUR NEXT MESSAGE HERE]")

In [ ]:
say("[YOUR NEXT MESSAGE HERE]", temperature=1.2)

To see the whole conversation at once, so you can copy it into your Journal:

In [ ]:
show_chat()

**NOTE:** This small AI is not very good at long conversations, and it loses the thread quickly.
If your specimen needs many turns, record what happens here and say so in your Journal.

## E. What to expect at each temperature

| Temperature | What usually happens |
|---|---|
| 0.01 | Almost the same answer every run |
| 0.5 | Small differences in wording |
| 0.9 | Clear variety, still sensible |
| 1.2 | More variety, more mistakes, sometimes odd |
| 1.5 | Often falls apart into nonsense |

Your results will not match this table exactly, and where your prompt stops making sense is worth
writing about.

## Handing it in

1. Copy your results into your team's Journal, in the tables from the project brief.
2. Share this notebook as an **Editor** with everyone in your team and the instructor.
3. Put the link to this notebook in your Journal, next to your tables.

**Acknowledgments:** Notebook created for the QRAI Mini-Project, in collaboration with Claude (see [conversation](https://claude.ai/share/654b2c64-ab66-4c34-9825-29678aa5856c)).

Current version created by Ethan C. Brown in collaboration with Claude Code.